In [1]:
"""Shared setup for all methods: data, ONE fixed train/test split, metrics, plots."""
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.optimize import linear_sum_assignment
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, cohen_kappa_score, precision_recall_fscore_support,
                             confusion_matrix)

CSV = "indian_pines.csv"
GT_CSV = "indian_pines_ground_truth.csv"
TRAIN_FRAC = 0.10            # 10% of each class for training - the SAME split for every method
SEED = 0
OUT = "results"
os.makedirs(OUT, exist_ok=True)

# ---------------- data
data = pd.read_csv(CSV)
gt = pd.read_csv(GT_CSV)
df = data.merge(gt[["row", "col", "label"]], on=["row", "col"], how="left")
band_cols = [c for c in df.columns if c.startswith("band_")]
X_all = df[band_cols].to_numpy(np.float32)             # every pixel (21025 x 200)
y_all = df["label"].fillna(0).astype(int).to_numpy()   # 0 = unlabelled
rows, cols = df["row"].to_numpy(), df["col"].to_numpy()
H, W = rows.max() + 1, cols.max() + 1
names = gt[gt["label"] > 0].drop_duplicates("label").set_index("label")["class_name"].sort_index()
classes = names.index.to_numpy()                        # 1..16

lab_idx = np.flatnonzero(y_all > 0)                     # CSV rows of the labelled pixels
X, y = X_all[lab_idx], y_all[lab_idx]


def split(y, frac, seed, min_per_class=5):
    """Stratified split: frac of each class (at least 5 pixels) for training."""
    rng = np.random.default_rng(seed)
    tr = np.zeros(len(y), bool)
    for c in np.unique(y):
        idx = np.flatnonzero(y == c)
        k = min(max(min_per_class, round(frac * len(idx))), len(idx) - 1)
        tr[rng.choice(idx, k, replace=False)] = True
    return tr


train = split(y, TRAIN_FRAC, SEED)
test = ~train
train_idx, test_idx = lab_idx[train], lab_idx[test]    # CSV rows of training / test pixels
print(f"{len(X_all)} pixels, {X.shape[1]} bands, {len(classes)} classes | train {train.sum()}, test {test.sum()}")


# ---------------- features
def pca_features(n_components):
    """PCA fitted on ALL pixels (no labels used). Returns (n_pixels, n_components)."""
    Z = StandardScaler().fit_transform(X_all)
    return PCA(n_components=n_components, random_state=SEED).fit_transform(Z).astype(np.float32)


def padded_cube(features_all, patch):
    """Rebuild the image (H, W, C) from per-pixel features and reflect-pad it for patch extraction."""
    cube = np.zeros((H, W, features_all.shape[1]), np.float32)
    cube[rows, cols] = features_all
    r = patch // 2
    return np.pad(cube, ((r, r), (r, r), (0, 0)), mode="reflect")


def windows(padded, idx, patch):
    """(N, patch, patch, C) windows centred on the pixels in CSV rows idx."""
    return np.stack([padded[rows[i]:rows[i] + patch, cols[i]:cols[i] + patch] for i in idx])


# ---------------- models
def tuned_svm(Xtr, ytr, standardize=True):
    """RBF-SVM; C and gamma chosen by 3-fold cross-validation on the training pixels only."""
    g0 = 1.0 / Xtr.shape[1] if standardize else 1.0 / (Xtr.shape[1] * Xtr.var())
    model = make_pipeline(StandardScaler(with_mean=standardize, with_std=standardize), SVC(kernel="rbf"))
    grid = {"svc__C": [10, 100, 1000], "svc__gamma": [g0 / 10, g0, g0 * 10]}
    cv = StratifiedKFold(3, shuffle=True, random_state=SEED)
    return GridSearchCV(model, grid, cv=cv, n_jobs=-1).fit(Xtr, ytr).best_estimator_


def clusters_to_classes(clusters_all):
    """Name each cluster after a class (Hungarian one-to-one matching on TRAINING pixels only).
    Leftover clusters (if more clusters than classes) get their majority class."""
    c_tr, y_tr = clusters_all[train_idx], y[train]
    K = clusters_all.max() + 1
    cont = np.zeros((K, len(classes)), int)
    for j, c in enumerate(classes):
        cont[:, j] = np.bincount(c_tr[y_tr == c], minlength=K)
    r_idx, c_idx = linear_sum_assignment(-cont)
    mapping = {r: classes[c] for r, c in zip(r_idx, c_idx)}
    for k in range(K):
        if k not in mapping:
            mapping[k] = classes[cont[k].argmax()] if cont[k].sum() > 0 else 0
    return np.array([mapping[k] for k in clusters_all])


def train_net(model, get_batch, n_train, y_train, epochs=100, lr=1e-3, batch=32):
    """PyTorch training loop. get_batch(positions) -> input tensor for those training samples."""
    import torch
    import torch.nn as nn
    torch.manual_seed(SEED)
    device = next(model.parameters()).device
    yt = torch.as_tensor(np.searchsorted(classes, y_train), dtype=torch.long)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()
    rng = np.random.default_rng(SEED)
    history = []
    for epoch in range(epochs):
        model.train()
        perm = rng.permutation(n_train)
        losses = []
        for i in range(0, n_train - batch // 2, batch):         # skips a tiny last batch (BatchNorm)
            b = perm[i:i + batch]
            xb, yb = get_batch(b).to(device), yt[torch.as_tensor(b)].to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        sched.step()
        history.append(float(np.mean(losses)))
        if (epoch + 1) % max(1, epochs // 10) == 0:
            print(f"  epoch {epoch + 1}/{epochs}  loss {history[-1]:.4f}")
    return history


def predict_net(model, n, get_batch, batch=512):
    """Predict n samples; get_batch(positions) -> input tensor. Returns class ids 1..16."""
    import torch
    device = next(model.parameters()).device
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, n, batch):
            out.append(model(get_batch(np.arange(i, min(i + batch, n))).to(device)).argmax(1).cpu().numpy())
    return classes[np.concatenate(out)]


# ---------------- evaluation + plots
_tab20 = plt.get_cmap("tab20").colors
CMAP = ListedColormap(["white"] + [_tab20[i] for i in list(range(0, 20, 2)) + list(range(1, 12, 2))])


def evaluate(method, y_pred_test, train_time, pred_all=None, extra=None):
    """Same metrics for every method, on the same test pixels. Saves json, per-class csv, confusion, map."""
    y_true = y[test]
    p, r, f1, support = precision_recall_fscore_support(y_true, y_pred_test, labels=classes, zero_division=0)
    res = {"Method": method,
           "OA (%)": 100 * accuracy_score(y_true, y_pred_test),
           "AA = Recall (%)": 100 * r.mean(),
           "Precision (%)": 100 * p.mean(),
           "F1 (%)": 100 * f1.mean(),
           "Kappa": cohen_kappa_score(y_true, y_pred_test),
           "Train time (s)": train_time}
    res.update(extra or {})
    print(f"\n===== {method} =====")
    for k, v in res.items():
        if k != "Method":
            print(f"{k:>18}: {v:.4f}")

    json.dump({k: (v if k == "Method" else float(v)) for k, v in res.items()},
              open(f"{OUT}/{method}_metrics.json", "w"), indent=2)
    np.save(f"{OUT}/{method}_test_pred.npy", np.asarray(y_pred_test))
    pd.DataFrame({"Class": [f"{c}. {names[c]}" for c in classes], "Precision (%)": (100 * p).round(1),
                  "Recall (%)": (100 * r).round(1), "F1 (%)": (100 * f1).round(1), "Test pixels": support}
                 ).to_csv(f"{OUT}/{method}_per_class.csv", index=False)

    cm = confusion_matrix(y_true, y_pred_test, labels=classes)
    pct = 100 * cm / cm.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(pct, cmap="Blues", vmin=0, vmax=100)
    for i in range(len(classes)):
        for j in range(len(classes)):
            if pct[i, j] >= 1:
                ax.text(j, i, f"{pct[i, j]:.0f}", ha="center", va="center", fontsize=6.5,
                        color="white" if pct[i, j] > 55 else "black")
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes)
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    ax.set_xlabel("Predicted class"); ax.set_ylabel("True class"); ax.set_title(method, loc="left")
    fig.colorbar(im, ax=ax, fraction=0.045, label="% of true-class pixels")
    fig.tight_layout()
    fig.savefig(f"{OUT}/{method}_confusion.png", dpi=300); fig.savefig(f"{OUT}/{method}_confusion.eps")
    plt.close(fig)

    if pred_all is not None:
        gt_img = np.zeros((H, W)); gt_img[rows, cols] = y_all
        pr_img = np.zeros((H, W)); pr_img[rows, cols] = pred_all
        fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))
        for a, img, t in zip(axes, (gt_img, pr_img), ("(a) Ground truth", f"(b) {method}")):
            a.imshow(img, cmap=CMAP, vmin=-0.5, vmax=16.5, interpolation="nearest")
            a.set_title(t, loc="left", fontsize=9); a.set_xticks([]); a.set_yticks([])
        fig.tight_layout()
        fig.savefig(f"{OUT}/{method}_map.png", dpi=300); fig.savefig(f"{OUT}/{method}_map.eps")
        plt.close(fig)
    return res


def plot_loss(history, method):
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.plot(range(1, len(history) + 1), history, lw=2)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Training loss"); ax.set_title(method, loc="left")
    ax.grid(color="#e4e3df")
    fig.tight_layout()
    fig.savefig(f"{OUT}/{method}_loss.png", dpi=300); fig.savefig(f"{OUT}/{method}_loss.eps")
    plt.close(fig)

21025 pixels, 200 bands, 16 classes | train 1030, test 9219


In [4]:
from common import *

N_LIST = [5, 10, 20, 30, 50]     # numbers of components to test with the SVM
N_KEEP = 30                      # this one goes into the master table

# ---------------- PCA on all pixels (no labels)
Z = StandardScaler().fit_transform(X_all)
pca = PCA(random_state=SEED).fit(Z)
scores = pca.transform(Z).astype(np.float32)
var = 100 * pca.explained_variance_ratio_
cum = np.cumsum(var)
print(pd.DataFrame({"PC": range(1, 11), "Variance (%)": var[:10].round(2), "Cumulative (%)": cum[:10].round(2)}).to_string(index=False))
for t in (90, 95, 99, 99.9):
    print(f"{t}% of the variance: {np.searchsorted(cum, t) + 1} PCs")
pd.DataFrame({"PC": range(1, len(var) + 1), "Variance (%)": var, "Cumulative (%)": cum}).round(4).to_csv(f"{OUT}/PCA_variance.csv", index=False)

# scree + cumulative variance
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.bar(range(1, 21), var[:20])
a1.set_xlabel("Principal component"); a1.set_ylabel("Explained variance (%)"); a1.set_title("(a) Scree plot", loc="left")
a2.plot(range(1, 61), cum[:60], marker="o", ms=3, lw=2)
for t in (95, 99):
    n = np.searchsorted(cum, t) + 1
    a2.axhline(t, ls=":", color="grey")
    a2.annotate(f"{t}% with {n} PCs", (n, t), xytext=(6, -12), textcoords="offset points", fontsize=8)
a2.set_xlabel("Number of components"); a2.set_ylabel("Cumulative variance (%)"); a2.set_title("(b) Cumulative variance", loc="left")
for a in (a1, a2):
    a.grid(color="#e4e3df")
fig.tight_layout(); fig.savefig(f"{OUT}/PCA_variance.png", dpi=300); fig.savefig(f"{OUT}/PCA_variance.eps"); plt.close(fig)

# loadings of PC1-3
fig, ax = plt.subplots(figsize=(7, 3.4))
for k in range(3):
    ax.plot(range(1, X_all.shape[1] + 1), pca.components_[k], lw=1.5, label=f"PC{k + 1}")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Band number"); ax.set_ylabel("Loading"); ax.legend(frameon=False); ax.grid(color="#e4e3df")
fig.tight_layout(); fig.savefig(f"{OUT}/PCA_loadings.png", dpi=300); fig.savefig(f"{OUT}/PCA_loadings.eps"); plt.close(fig)

# PC images
fig, axes = plt.subplots(1, 4, figsize=(12, 3.3))
for k, a in enumerate(axes):
    img = np.zeros((H, W)); img[rows, cols] = scores[:, k]
    a.imshow(img, cmap="gray"); a.set_title(f"PC{k + 1} ({var[k]:.1f}%)", loc="left", fontsize=9)
    a.set_xticks([]); a.set_yticks([])
fig.tight_layout(); fig.savefig(f"{OUT}/PCA_images.png", dpi=300); fig.savefig(f"{OUT}/PCA_images.eps"); plt.close(fig)

# ---------------- PCA + SVM (PC scores are not re-standardised, so PC1 keeps more weight than PC50)
curve = []
for n in N_LIST:
    F = scores[:, :n]
    t0 = time.time()
    model = tuned_svm(F[train_idx], y[train], standardize=False)
    t_train = time.time() - t0
    y_pred = model.predict(F[test_idx])
    curve.append({"PCs": n, "OA (%)": 100 * accuracy_score(y[test], y_pred)})
    print(f"{n:>3} PCs: OA = {curve[-1]['OA (%)']:.2f} %")
    if n == N_KEEP:
        evaluate(f"PCA{N_KEEP}-SVM", y_pred, t_train, pred_all=model.predict(F))
curve = pd.DataFrame(curve)
curve.round(3).to_csv(f"{OUT}/PCA_SVM_curve.csv", index=False)

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(curve["PCs"], curve["OA (%)"], marker="o", lw=2)
ax.set_xlabel("Number of principal components"); ax.set_ylabel("Overall accuracy (%)"); ax.grid(color="#e4e3df")
fig.tight_layout(); fig.savefig(f"{OUT}/PCA_SVM_curve.png", dpi=300); fig.savefig(f"{OUT}/PCA_SVM_curve.eps"); plt.close(fig)

 PC  Variance (%)  Cumulative (%)
  1     68.620003       68.620003
  2     19.190001       87.809998
  3      2.560000       90.370003
  4      1.690000       92.059998
  5      1.050000       93.110001
  6      0.840000       93.949997
  7      0.470000       94.419998
  8      0.390000       94.809998
  9      0.370000       95.190002
 10      0.350000       95.529999
90% of the variance: 3 PCs
95% of the variance: 9 PCs
99% of the variance: 40 PCs
99.9% of the variance: 108 PCs
  5 PCs: OA = 68.86 %
 10 PCs: OA = 72.56 %
 20 PCs: OA = 76.80 %
 30 PCs: OA = 78.50 %

===== PCA30-SVM =====
            OA (%): 78.5009
   AA = Recall (%): 76.8195
     Precision (%): 76.1819
            F1 (%): 76.1562
             Kappa: 0.7542
    Train time (s): 0.9845
 50 PCs: OA = 79.78 %


In [5]:
from common import *

t0 = time.time()
model = tuned_svm(X[train], y[train])            # standardise + tune C and gamma
train_time = time.time() - t0
svc = model.named_steps["svc"]
print(f"best C = {svc.C}, gamma = {svc.gamma:.4g}")

y_pred = model.predict(X[test])
pred_all = model.predict(X_all)                  # whole scene, for the map
evaluate("SVM", y_pred, train_time, pred_all=pred_all)

best C = 100, gamma = 0.005

===== SVM =====
            OA (%): 81.4188
   AA = Recall (%): 78.8245
     Precision (%): 79.5029
            F1 (%): 78.9488
             Kappa: 0.7876
    Train time (s): 1.4353


{'Method': 'SVM',
 'OA (%)': 81.41880898145135,
 'AA = Recall (%)': np.float64(78.82448926498277),
 'Precision (%)': np.float64(79.50294420211314),
 'F1 (%)': np.float64(78.94884276793081),
 'Kappa': 0.7876170487536046,
 'Train time (s)': 1.4352600574493408}

In [6]:
from common import *
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, davies_bouldin_score

N_PCS = 20
K = 16                       # one cluster per class
COV = "full"
RUN_BIC = True               # BIC curve for choosing K (takes a few minutes)

F = pca_features(N_PCS)

t0 = time.time()
gmm = GaussianMixture(n_components=K, covariance_type=COV, n_init=5, random_state=SEED).fit(F)
clusters = gmm.predict(F)                        # no labels used
train_time = time.time() - t0

mapped = clusters_to_classes(clusters)           # cluster -> class names (training pixels only)
extra = {"ARI": adjusted_rand_score(y[test], clusters[test_idx]),
         "NMI": normalized_mutual_info_score(y[test], clusters[test_idx]),
         "Silhouette": silhouette_score(F, clusters, sample_size=5000, random_state=SEED),
         "Davies-Bouldin": davies_bouldin_score(F, clusters)}
evaluate("GMM", mapped[test_idx], train_time, pred_all=mapped, extra=extra)

if RUN_BIC:
    ks = list(range(4, 31, 2))
    bic = [GaussianMixture(k, covariance_type=COV, random_state=SEED).fit(F).bic(F) for k in ks]
    fig, ax = plt.subplots(figsize=(5.5, 3.4))
    ax.plot(ks, bic, marker="o", lw=2)
    ax.axvline(ks[int(np.argmin(bic))], color="grey", ls=":")
    ax.set_xlabel("Number of clusters"); ax.set_ylabel("BIC (lower = better)"); ax.grid(color="#e4e3df")
    fig.tight_layout(); fig.savefig(f"{OUT}/GMM_bic.png", dpi=300); fig.savefig(f"{OUT}/GMM_bic.eps"); plt.close(fig)
    print("BIC minimum at k =", ks[int(np.argmin(bic))])


===== GMM =====
            OA (%): 32.1293
   AA = Recall (%): 34.5281
     Precision (%): 31.2889
            F1 (%): 28.0132
             Kappa: 0.2562
    Train time (s): 34.5660
               ARI: 0.1782
               NMI: 0.3773
        Silhouette: 0.1085
    Davies-Bouldin: 1.6625
BIC minimum at k = 30


In [7]:
from common import *
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score, silhouette_score,
                             davies_bouldin_score, pairwise_distances_argmin)

N_PCS = 20
K = 16
N_SAMPLE = 5000              # tree built on a sample (all 21k pixels would need ~2 GB RAM)
METHOD = "ward"

F = pca_features(N_PCS)
sample = np.random.default_rng(SEED).choice(len(F), N_SAMPLE, replace=False)
S = F[sample]

t0 = time.time()
L = linkage(S, method=METHOD)
lab_s = fcluster(L, t=K, criterion="maxclust") - 1
centroids = np.array([S[lab_s == c].mean(0) for c in np.unique(lab_s)])
clusters = pairwise_distances_argmin(F, centroids)   # every pixel -> nearest cluster centre
train_time = time.time() - t0

mapped = clusters_to_classes(clusters)
extra = {"ARI": adjusted_rand_score(y[test], clusters[test_idx]),
         "NMI": normalized_mutual_info_score(y[test], clusters[test_idx]),
         "Silhouette": silhouette_score(F, clusters, sample_size=5000, random_state=SEED),
         "Davies-Bouldin": davies_bouldin_score(F, clusters)}
evaluate("Hierarchical", mapped[test_idx], train_time, pred_all=mapped, extra=extra)

# dendrogram with the cut that gives K clusters
cut = (L[-(K - 1), 2] + L[-K, 2]) / 2
fig, ax = plt.subplots(figsize=(8, 3.8))
dendrogram(L, truncate_mode="lastp", p=40, color_threshold=cut, above_threshold_color="grey", no_labels=True, ax=ax)
ax.axhline(cut, color="black", ls="--", lw=1)
ax.set_ylabel("Merge distance"); ax.set_title(f"Dendrogram ({METHOD}), cut at k = {K}", loc="left")
fig.tight_layout(); fig.savefig(f"{OUT}/Hierarchical_dendrogram.png", dpi=300); fig.savefig(f"{OUT}/Hierarchical_dendrogram.eps"); plt.close(fig)


===== Hierarchical =====
            OA (%): 41.5555
   AA = Recall (%): 36.5280
     Precision (%): 39.8620
            F1 (%): 36.8489
             Kappa: 0.3399
    Train time (s): 2.2939
               ARI: 0.2636
               NMI: 0.4563
        Silhouette: 0.2470
    Davies-Bouldin: 1.1510


In [8]:
from common import *
import torch
import torch.nn as nn

EPOCHS = 200
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

scaler = StandardScaler().fit(X[train])                       # training statistics only
Xtr = torch.tensor(scaler.transform(X[train]), dtype=torch.float32).unsqueeze(1)   # (N, 1, 200)


def rows_to_input(idx_rows):
    return torch.tensor(scaler.transform(X_all[idx_rows]), dtype=torch.float32).unsqueeze(1)


class CNN1D(nn.Module):
    def __init__(self, n_bands, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2))
        self.classifier = nn.Sequential(                       # flatten keeps band position
            nn.Flatten(), nn.Dropout(0.5), nn.Linear(64 * (n_bands // 8), 128), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


model = CNN1D(X.shape[1], len(classes)).to(device)
n_params = sum(p.numel() for p in model.parameters())

t0 = time.time()
history = train_net(model, lambda b: Xtr[torch.as_tensor(b)], len(train_idx), y[train], epochs=EPOCHS)
train_time = time.time() - t0

y_pred = predict_net(model, len(test_idx), lambda b: rows_to_input(test_idx[b]))
pred_all = predict_net(model, len(X_all), lambda b: rows_to_input(b), batch=2048)
evaluate("1D-CNN", y_pred, train_time, pred_all=pred_all, extra={"Parameters": n_params})
plot_loss(history, "1D-CNN")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\AES\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\AES\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\AES\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\AES\AppData\Roaming\Python\Python310\site-packages\traitlets\config\application.py", l

RuntimeError: Could not infer dtype of numpy.int64